# Part 0 - Pre-quantize Llama-2-7B-chat to INT8 (GPTQModel)

Loads `meta-llama/Llama-2-7b-chat-hf` once, applies **GPTQModel** quantization
with `bits=8, group_size=128` (recipe from
[vLLM's GPTQModel docs](https://docs.vllm.ai/en/latest/features/quantization/gptqmodel/)),
and saves the resulting GPTQ checkpoint to `./quantized_int8/`. All later
notebooks load **from this local path** instead of from HF via the path-rewriting
layer in `jbb_common.patch_jbb()`. vLLM auto-detects the GPTQ format from
`config.json` - no `quantization=` kwarg needed at load time, and the Marlin /
Machete kernels accelerate inference on Ampere+ GPUs.

Calibration uses 1024 samples from `allenai/c4` (the official recipe). Quantize
step takes a few minutes on a single GPU; loading is fast afterwards.

Run this **first**, in its own kernel. Idempotent: re-running with the
checkpoint already on disk is a no-op.

**Output**: `./quantized_int8/` (model + tokenizer files).


In [ ]:
# Install gptqmodel per vLLM docs: https://docs.vllm.ai/en/latest/features/quantization/gptqmodel/
# Three pre-steps for the DSMLP cluster (Python 3.13, no pre-built gptqmodel wheel):
#   1. setuptools >= 77 so it accepts gptqmodel 7.0's PEP 639 SPDX license string
#      ("Apache-2.0" / oneOf error).
#   2. transformers >= 4.51 so `transformers.utils.hub.create_repo` is re-exported
#      as a module attribute (gptqmodel 7.0 reads it that way; older transformers
#      raises AttributeError on import).
#   3. --no-build-isolation so the from-source build sees the installed torch
#      (gptqmodel compiles CUDA extensions against it).
import importlib, subprocess, sys
if importlib.util.find_spec("gptqmodel") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--user", "--upgrade",
        "setuptools>=77", "packaging>=24", "wheel",
        "transformers>=4.51",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--user", "-U",
        "gptqmodel", "--no-build-isolation",
    ])
    print("Installed gptqmodel. RESTART THE KERNEL before running the next cell so")
    print("the upgraded transformers / setuptools / gptqmodel are picked up.")
else:
    print("gptqmodel already installed.")


In [ ]:
import os, sys
from pathlib import Path

# Self-locate jbb_common.py so kernel CWD doesn't matter.
_cwd = Path.cwd()
if (_cwd / "jbb_common.py").exists():
    pass
elif (_cwd / "tml" / "jbb_common.py").exists():
    os.chdir(_cwd / "tml")
else:
    raise FileNotFoundError(f"jbb_common.py not found from {_cwd}; run from tml/ or repo root.")
sys.path.insert(0, str(Path.cwd()))

from jbb_common import hf_login
hf_login()


In [ ]:
# Pre-quantize Llama-2-7B-chat-hf with GPTQModel and save locally.
# Recipe copied directly from vLLM's GPTQModel docs (bits=8, group_size=128).
# vLLM's Marlin / Machete kernels accelerate GPTQ inference on Ampere+ GPUs.

import os
import gc
import torch
from pathlib import Path
from datasets import load_dataset
from gptqmodel import GPTQModel, QuantizeConfig
from transformers import AutoTokenizer

# Force PyTorch and Hugging Face to use a single thread for data loading
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
# Prevent deadlocks in Python multiprocessing forks
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SRC_MODEL  = "meta-llama/Llama-2-7b-chat-hf"
QUANT_PATH = Path("./quantized_int8")

# Idempotency guard: require BOTH config.json AND at least one weight shard
# AND tokenizer.model. The original `config.json`-only check let a half-written
# bnb stub (config-but-no-weights) survive and silently break every downstream
# notebook. This stricter check forces a re-quantize if anything is missing.
_has_config  = (QUANT_PATH / "config.json").exists()
_has_weights = any(QUANT_PATH.glob("*.safetensors")) or any(QUANT_PATH.glob("*.bin"))
_has_tokenizer = (QUANT_PATH / "tokenizer.model").exists()

if _has_config and _has_weights and _has_tokenizer:
    print(f"Already quantized at {QUANT_PATH.resolve()} - skipping.")
else:
    if _has_config and not (_has_weights and _has_tokenizer):
        print(f"Stale/incomplete checkpoint at {QUANT_PATH.resolve()} "
              f"(weights={_has_weights}, tokenizer={_has_tokenizer}). Re-quantizing.")
        import shutil
        shutil.rmtree(QUANT_PATH, ignore_errors=True)

    calibration_dataset = load_dataset(
        "allenai/c4",
        data_files="en/c4-train.00001-of-01024.json.gz",
        split="train",
    ).select(range(256)) # 1024 -> 256

    quant_config = QuantizeConfig(bits=8, group_size=128)

    model = GPTQModel.load(SRC_MODEL, quant_config)
    model.quantize(calibration_dataset, batch_size=1) # 2 -> 1
    model.save(str(QUANT_PATH))

    # Save tokenizer alongside weights so downstream notebooks can load the
    # checkpoint fully offline. GPTQModel.save() doesn't emit the SPM vocab,
    # so without this `LLM(model="./quantized_int8")` would require an HF
    # login + gated-repo access for the tokenizer at runtime.
    AutoTokenizer.from_pretrained(SRC_MODEL).save_pretrained(str(QUANT_PATH))

    print(f"Saved INT8 GPTQ checkpoint + tokenizer to {QUANT_PATH.resolve()}")

    del model
    gc.collect()
    torch.cuda.empty_cache()